In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
from shapely.geometry import Polygon
import re

# 1. Load the Data
gdb_path = "alameda_Parcel_Ownership_20260915.gdb"
gdf = gpd.read_file(gdb_path, layer="PARCEL_With_Ownership")

# 2. Define Categorization Functions
def categorize_ownership(row):
    use_code = str(row.get('usecode', '')).strip()
    owner = str(row.get('ownername', '')).upper()
    
    # Public Entities
    if use_code in ['300', '6001', '6100']:
        if 'UNITED STATES' in owner or 'US ' in owner: return 'Public - Federal'
        if 'STATE OF CA' in owner: return 'Public - State'
        if 'COUNTY' in owner: return 'Public - County'
        if 'CITY' in owner or 'OAKLAND' in owner: return 'Public - City'
        return 'Public - Other'
        
    # Corporate & Trust (Overrides standard residential if owned by entity)
    if re.search(r'\b(LLC|INC|CORP|LTD|LP)\b', owner):
        return 'Corporate/Investor'
    if re.search(r'\b(TRUST|TR)\b', owner):
        return 'Trust'
        
    # Communal / Condo
    if use_code.startswith('73') or use_code.startswith('15') or use_code.startswith('16') or use_code == '3900' or use_code == '4101':
        return 'Condo/Communal'
        
    # Individual / Single Family
    if use_code == '1100' or use_code.startswith('2'):
        return 'Individual/Small Multi-Family'
        
    return 'Other/Commercial'

# Apply categorization
gdf['owner_category'] = gdf.apply(categorize_ownership, axis=1)

# 3. Create the Northern Waterfront Boundary
# Fetching street networks from OSM to create a precise bounding polygon
# (Alternatively, you can replace this with hardcoded EPSG:2227 coordinates from your GIS)
place = "Alameda, California, USA"
graph = ox.graph_from_place(place, network_type='drive')
edges = ox.graph_to_gdfs(graph, nodes=False, edges=True)

webster = edges[edges['name'] == 'Webster Street']
central = edges[edges['name'] == 'Central Avenue']
willow = edges[edges['name'] == 'Willow Street']

# In a full GIS workflow, you would extract the envelope of these merged streets.
# For simplicity in this script, we'll define the bounding box using approximate WGS84 coords
# and project them to match the GDB (EPSG:2227).
nw_polygon_wgs84 = Polygon([
    (-122.277, 37.771), # NW Corner (Webster at Estuary)
    (-122.250, 37.771), # NE Corner (Willow at Estuary)
    (-122.250, 37.763), # SE Corner (Willow at Central)
    (-122.277, 37.763)  # SW Corner (Webster at Central)
])

# Create a GeoDataFrame for the boundary and project it
boundary_gdf = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[nw_polygon_wgs84])
boundary_gdf = boundary_gdf.to_crs(gdf.crs)

# 4. Spatial Join/Flagging
# Create a boolean column marking if a parcel intersects the Northern Waterfront boundary
gdf['in_northern_waterfront'] = gdf.intersects(boundary_gdf.geometry.iloc[0])

# 5. Clean up missing values and write to GeoParquet
gdf['yearbuilt'] = gdf['yearbuilt'].replace(0, pd.NA)
gdf['buildingarea'] = gdf['buildingarea'].replace(0, pd.NA)

output_path = "alameda_parcels_cleaned.parquet"
gdf.to_parquet(output_path, index=False)
print(f"Data successfully cleaned and exported to {output_path}")

ModuleNotFoundError: No module named 'osmnx'